### Bland Altman + deming regression

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "sensors").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not find project root.")


project_root = find_project_root(Path.cwd())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.visualization.bland_altmant_plot import plot_bland_altman
from src.time_series.time_series_comparison_metrics import compute_deming_regression
from src.time_series.time_series_sampler import resample_and_align

SENSORS_DIR = project_root / "data" / "sensors"
FIGURES_DIR = project_root / "data" / "docs_reports"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

data_test_path = SENSORS_DIR / "boston_carter_field" / "boston_carter_field.csv"
data_reference_path = SENSORS_DIR / "boston_carter_field_reference" / "MOD-00157-CarterField.csv"

data_test = pd.read_csv(data_test_path)
data_reference = pd.read_csv(data_reference_path)

aligned = resample_and_align(
    data_test,
    data_reference,
    time_column="timestamp",
    variable_name="temperature",
    sampling_interval="0.1min",
)
print(f"Aligned rows: {len(aligned)}")
aligned.head()

#### Temperature

In [ ]:
temp_series_1 = list(aligned["temperature_1_aligned"])
temp_series_2 = list(aligned["temperature_2"])

plot_bland_altman(
    temp_series_1, temp_series_2,
    title="Bland-Altman (temperature)",
    xlabel="Mean Temperature (C)",
    ylabel="Difference in Temperature (C)",
)
plt.savefig(FIGURES_DIR / "temperature_bland_altman.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from src.time_series.time_series_comparison_metrics import compute_deming_regression

slope, intercept = compute_deming_regression(
    temp_series_2, temp_series_1,
    title="Deming Regression (temperature)",
    xlabel="Reference Temperature (C)",
    ylabel="Test Temperature (C)",
)
plt.savefig(FIGURES_DIR / "temperature_deming_regression.png", dpi=300, bbox_inches="tight")
plt.show()

### PM 2.5

In [ ]:
aligned = resample_and_align(
    data_test,
    data_reference,
    time_column="timestamp",
    variable_name="pm25",
    sampling_interval="0.1min",
)

pm25_test = list(aligned["pm25_1_aligned"])
pm25_ref = list(aligned["pm25_2"])

plot_bland_altman(
    pm25_test, pm25_ref,
    title="Bland-Altman (PM2.5)",
    xlabel="Mean PM2.5 (µg/m³)",
    ylabel="Difference in PM2.5 (µg/m³)",
)
plt.savefig(FIGURES_DIR / "pm2.5_bland_altman.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
slope, intercept = compute_deming_regression(
    pm25_ref, pm25_test,
    title="Deming Regression (PM2.5)",
    xlabel="Reference PM2.5 (µg/m³)",
    ylabel="Test PM2.5 (µg/m³)",
)
plt.savefig(FIGURES_DIR / "pm2.5_deming_regression.png", dpi=300, bbox_inches="tight")
plt.show()

### Humidity

In [ ]:
data_reference["relative_humidity"] = data_reference["rh"]
aligned = resample_and_align(
    data_test,
    data_reference,
    time_column="timestamp",
    variable_name="relative_humidity",
    sampling_interval="0.1min",
)

rh_series_1 = list(aligned["relative_humidity_1_aligned"])
rh_series_2 = list(aligned["relative_humidity_2"])

plot_bland_altman(
    rh_series_1, rh_series_2,
    title="Bland-Altman (Relative Humidity)",
    xlabel="Mean Relative Humidity (%)",
    ylabel="Difference in Relative Humidity (%)",
)
plt.savefig(FIGURES_DIR / "relative_humidity_bland_altman.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
slope, intercept = compute_deming_regression(
    rh_series_2, rh_series_1,
    title="Deming Regression (Relative Humidity)",
    xlabel="Reference Relative Humidity (%)",
    ylabel="Test Relative Humidity (%)",
)
plt.savefig(FIGURES_DIR / "relative_humidity_deming_regression.png", dpi=300, bbox_inches="tight")
plt.show()